# 02 — Làm sạch Pareto Front & Common Fleet Target

Thực hiện **Section 4** (clean pareto front) và **Section 5** (common fleet
target + Distance-Focus solution) của kế hoạch phân tích.

Input: `artifacts/run_manifest_matched_valid.csv` (từ notebook 01).
Output: `artifacts/solutions_clean.csv`, `artifacts/fleet_targets.csv`,
`artifacts/td_solutions.csv`.


In [1]:
# ==== CẤU HÌNH ĐƯỜNG DẪN (chỉnh lại cho đúng máy của bạn) ====
import sys, os
sys.path.append(os.path.abspath("."))  # để import evrp_analysis_utils.py cùng thư mục

FULL_DIR  = "benchmark/full"     # thư mục kết quả bản đầy đủ (equity-aware)
NOEQ_DIR  = "benchmark/no-EQ"    # thư mục kết quả bản loại equity guidance
ARTIFACT_DIR = "artifacts"       # nơi lưu các bảng trung gian (csv) giữa các notebook
os.makedirs(ARTIFACT_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import evrp_analysis_utils as utils

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [2]:
manifest = pd.read_csv(f"{ARTIFACT_DIR}/run_manifest_matched_valid.csv")
print(f"{len(manifest)} runs matched+valid nạp từ notebook 01")
manifest.head()


1840 runs matched+valid nạp từ notebook 01


,Variant,Instance,Seed,RunDir,ConfigHash,Runtime_s,Iterations,FinalArchiveSize_reported,FrontObjRows,HasFrontObjectives,HasFrontDetails,HasEvolutionLog,HasProgress,HasOperators,Valid,Group,Size
0,FULL,c101_21,1,benchmark\full\c101_21\seed_1,d8f9b899d6,775.317,25000.0,30.0,30,True,True,True,True,True,True,C,21.0
1,FULL,c101_21,10,benchmark\full\c101_21\seed_10,bbac9020ed,737.186,25000.0,32.0,32,True,True,True,True,True,True,C,21.0
2,FULL,c101_21,2,benchmark\full\c101_21\seed_2,6a0f35a2bd,800.131,25000.0,43.0,43,True,True,True,True,True,True,C,21.0
3,FULL,c101_21,3,benchmark\full\c101_21\seed_3,29f6a8f112,747.130,25000.0,28.0,28,True,True,True,True,True,True,C,21.0
4,FULL,c101_21,4,benchmark\full\c101_21\seed_4,0dfca08bd7,778.205,25000.0,28.0,28,True,True,True,True,True,True,C,21.0


## 2.1. Đọc toàn bộ `_front_objectives.csv` -> solutions_raw

In [3]:
raw_rows = []
for _, run in manifest.iterrows():
    path = os.path.join(run["RunDir"], f"{run['Instance']}_seed_{int(run['Seed'])}_front_objectives.csv")
    fo = utils.parse_front_objectives(path)
    if fo.empty:
        continue
    fo = fo.copy()
    fo["Variant"] = run["Variant"]
    fo["Instance"] = run["Instance"]
    fo["Seed"] = run["Seed"]
    fo["SolutionID"] = [f"{run['Instance']}_s{int(run['Seed'])}_{i}" for i in range(len(fo))]
    raw_rows.append(fo)

solutions_raw = pd.concat(raw_rows, ignore_index=True) if raw_rows else pd.DataFrame()
solutions_raw.to_csv(f"{ARTIFACT_DIR}/solutions_raw.csv", index=False)
print(f"Tổng số điểm Pareto thô: {len(solutions_raw)}")
solutions_raw.head()


Tổng số điểm Pareto thô: 36837


,SolutionID,Z1,Z2,Z3,Z4,IsFeasible,Variant,Instance,Seed
0,c101_21_s1_0,12,1060.8056,0.0504,1204.8760,True,FULL,c101_21,1
1,c101_21_s1_1,12,1056.1877,0.0547,1204.8760,True,FULL,c101_21,1
2,c101_21_s1_2,12,1064.2467,0.0491,1204.8760,True,FULL,c101_21,1
3,c101_21_s1_3,12,1065.4846,0.0533,1204.3369,True,FULL,c101_21,1
4,c101_21_s1_4,12,1061.9066,0.0468,1205.1856,True,FULL,c101_21,1


## 2.2. Lưu ý khác biệt objective vector giữa FULL và No-EQ

FULL dùng (Z2, Z3, Z4) làm hướng dẫn equity; No-EQ có thể chỉ dùng (Z2, Z4).
-> FinalArchiveSize không được dùng làm evidence chất lượng trực tiếp (đúng như tài liệu Section 4).

In [4]:
z_cols_present = [c for c in ["Z1", "Z2", "Z3", "Z4"] if c in solutions_raw.columns]
print("Các cột objective tìm thấy:", z_cols_present)

archive_size_by_variant = solutions_raw.groupby("Variant").size()
print("\nSố điểm archive thô theo variant (KHÔNG dùng làm evidence chất lượng):")
print(archive_size_by_variant)


Các cột objective tìm thấy: ['Z1', 'Z2', 'Z3', 'Z4']

Số điểm archive thô theo variant (KHÔNG dùng làm evidence chất lượng):
Variant
FULL     32096
No-EQ     4741
dtype: int64


## 2.3. Clean: loại infeasible -> loại trùng lặp -> dedupe gần giống -> nondominance

In [5]:
OBJ_COLS = [c for c in ["Z1", "Z2", "Z3", "Z4"] if c in solutions_raw.columns]

clean_frames = []
for (variant, instance, seed), g in solutions_raw.groupby(["Variant", "Instance", "Seed"]):
    cleaned = utils.clean_pareto_front(g, OBJ_COLS)
    clean_frames.append(cleaned)

solutions_clean = pd.concat(clean_frames, ignore_index=True) if clean_frames else pd.DataFrame()
solutions_clean.to_csv(f"{ARTIFACT_DIR}/solutions_clean.csv", index=False)

print(f"Trước clean: {len(solutions_raw)} điểm")
print(f"Sau clean (per run, nondominated): {len(solutions_clean)} điểm")
solutions_clean.groupby("Variant").size()


Trước clean: 36837 điểm
Sau clean (per run, nondominated): 29858 điểm


Variant
FULL     25117
No-EQ     4741
dtype: int64

## 2.4. Common Fleet Target Z1_common mỗi instance (Section 5)

$Z_{1,i}^{common} = \min\{k : \Pr_{FULL}(Z_1=k)\ge 0.5,\ \Pr_{NoEQ}(Z_1=k)\ge 0.5\}$

In [6]:
seeds_per_variant_instance = (
    manifest.groupby(["Variant", "Instance"])["Seed"].nunique().reset_index(name="NumSeeds")
)

fleet_rows = []
for instance, g_inst in solutions_clean.groupby("Instance"):
    front_full = g_inst[g_inst["Variant"] == "FULL"]
    front_noeq = g_inst[g_inst["Variant"] == "No-EQ"]

    n_full = seeds_per_variant_instance.query("Variant=='FULL' and Instance==@instance")["NumSeeds"]
    n_noeq = seeds_per_variant_instance.query("Variant=='No-EQ' and Instance==@instance")["NumSeeds"]
    n_full = int(n_full.iloc[0]) if len(n_full) else 0
    n_noeq = int(n_noeq.iloc[0]) if len(n_noeq) else 0

    z1_common = utils.common_fleet_target(front_full, front_noeq, n_full, n_noeq) if "Z1" in g_inst.columns else None
    fleet_rows.append({"Instance": instance, "Z1_common": z1_common, "NumSeeds_FULL": n_full, "NumSeeds_NoEQ": n_noeq})

fleet_targets = pd.DataFrame(fleet_rows)
fleet_targets.to_csv(f"{ARTIFACT_DIR}/fleet_targets.csv", index=False)

print(f"Instances không tìm được common fleet target (loại khỏi conditional-quality analysis): "
      f"{fleet_targets['Z1_common'].isna().sum()} / {len(fleet_targets)}")
fleet_targets.head(15)


Instances không tìm được common fleet target (loại khỏi conditional-quality analysis): 1 / 92


,Instance,Z1_common,NumSeeds_FULL,NumSeeds_NoEQ
0,c101C10,3.0,10,10
1,c101C5,2.0,10,10
2,c101_21,12.0,10,10
3,c102_21,11.0,10,10
4,c103C15,3.0,10,10
5,c103C5,1.0,10,10
6,c103_21,NaN,10,10
7,c104C10,2.0,10,10
8,c104_21,10.0,10,10
9,c105_21,11.0,10,10


## 2.5. Fleet attainment success rate (SR)

$SR_{i,v} = \dfrac{\#\{seed\ đạt\ Z_{1,i}^{common}\}}{10}$

In [7]:
sr_rows = []
merged = solutions_clean.merge(fleet_targets[["Instance", "Z1_common"]], on="Instance", how="left")
for (variant, instance), g in merged.groupby(["Variant", "Instance"]):
    z1c = g["Z1_common"].iloc[0]
    if pd.isna(z1c):
        continue
    n_seeds = int(seeds_per_variant_instance.query("Variant==@variant and Instance==@instance")["NumSeeds"].iloc[0])
    seeds_hit = g.loc[g["Z1"] == z1c, "Seed"].nunique()
    sr_rows.append({"Variant": variant, "Instance": instance, "Z1_common": z1c,
                     "SeedsHit": seeds_hit, "NumSeeds": n_seeds, "SR": seeds_hit / n_seeds if n_seeds else np.nan})

fleet_attainment = pd.DataFrame(sr_rows)
fleet_attainment.to_csv(f"{ARTIFACT_DIR}/fleet_attainment.csv", index=False)

print("Fleet attainment trung bình theo variant:")
print(fleet_attainment.groupby("Variant")["SR"].mean())
fleet_attainment.head(10)


Fleet attainment trung bình theo variant:
Variant
FULL     0.963736
No-EQ    0.965934
Name: SR, dtype: float64


,Variant,Instance,Z1_common,SeedsHit,NumSeeds,SR
0,FULL,c101C10,3.0,10,10,1.0
1,FULL,c101C5,2.0,10,10,1.0
2,FULL,c101_21,12.0,10,10,1.0
3,FULL,c102_21,11.0,10,10,1.0
4,FULL,c103C15,3.0,10,10,1.0
5,FULL,c103C5,1.0,10,10,1.0
6,FULL,c104C10,2.0,10,10,1.0
7,FULL,c104_21,10.0,10,10,1.0
8,FULL,c105_21,11.0,10,10,1.0
9,FULL,c106C15,3.0,10,10,1.0


## 2.6. Distance-Focus solution $s^{TD}$ tại common fleet target (Section 6.1)

$s^{TD} = \arg\min_{s: Z_1 = Z_1^{common}} Z_2(s)$ — lấy trên nghiệm gộp từ TẤT CẢ seed của mỗi (variant, instance).

In [8]:
td_rows = []
for (variant, instance), g in merged.groupby(["Variant", "Instance"]):
    z1c = g["Z1_common"].iloc[0]
    if pd.isna(z1c):
        continue
    td = utils.distance_focus_solution(g, int(z1c))
    if td is None:
        continue
    row = td.to_dict()
    row["Variant"] = variant
    row["Instance"] = instance
    td_rows.append(row)

td_solutions = pd.DataFrame(td_rows)
td_solutions.to_csv(f"{ARTIFACT_DIR}/td_solutions.csv", index=False)
print(f"{len(td_solutions)} Distance-Focus solutions (một mỗi variant x instance có common fleet target)")
td_solutions.head(10)


182 Distance-Focus solutions (một mỗi variant x instance có common fleet target)


,SolutionID,Z1,Z2,Z3,Z4,IsFeasible,Variant,Instance,Seed,Z1_common
0,c101C10_s1_5,3,388.2454,0.1195,1162.8058,True,FULL,c101C10,1,3.0
1,c101C5_s1_1,2,257.7475,0.1065,872.0789,True,FULL,c101C5,1,2.0
2,c101_21_s7_31,12,1044.5000,0.0921,1199.3839,True,FULL,c101_21,7,12.0
3,c102_21_s2_18,11,1041.9537,0.0781,1215.2132,True,FULL,c102_21,2,11.0
4,c103C15_s3_11,3,374.4278,0.0668,1175.6675,True,FULL,c103C15,3,3.0
5,c103C5_s1_1,1,175.3692,0.0000,1197.4184,True,FULL,c103C5,1,1.0
6,c104C10_s1_1,2,273.9312,0.2456,1185.9421,True,FULL,c104C10,1,2.0
7,c104_21_s2_24,10,904.5044,0.0355,1198.8609,True,FULL,c104_21,2,10.0
8,c105_21_s2_21,11,1058.4692,0.0740,1222.5025,True,FULL,c105_21,2,11.0
9,c106C15_s1_0,3,275.1332,0.2220,1082.9662,True,FULL,c106C15,1,3.0


## 2.7. So sánh sơ bộ TD solutions: FULL vs No-EQ

In [9]:
pivot_cols = [c for c in ["Z2", "Z3", "Z4"] if c in td_solutions.columns]
td_pivot = td_solutions.pivot_table(index="Instance", columns="Variant", values=pivot_cols)
td_pivot.head(15)


Z2                 Z3                 Z4           
Variant        FULL      No-EQ    FULL   No-EQ       FULL      No-EQ
Instance                                                            
c101C10    388.2454   388.2454  0.1195  0.1288  1162.8058  1162.8058
c101C5     257.7475   257.7475  0.1065  0.1181   872.0789   872.0789
c101_21   1044.5000  1043.4548  0.0921  0.1078  1199.3839  1199.3839
c102_21   1041.9537  1037.1225  0.0781  0.0652  1215.2132  1205.1121
c103C15    374.4278   374.4368  0.0668  0.1115  1175.6675  1175.6675
c103C5     175.3692   175.3692  0.0000  0.0000  1197.4184  1197.4184
c104C10    273.9312   273.9312  0.2456  0.2483  1185.9421  1185.9421
c104_21    904.5044   895.6013  0.0355  0.0536  1198.8609  1232.6795
c105_21   1058.4692  1052.1882  0.0740  0.0442  1222.5025  1174.3839
c106C15    275.1332   275.1332  0.2220  0.2248  1082.9662  1082.9662
c106_21   1031.5689  1015.8785  0.0867  0.0926  1223.0084  1233.7597
c107_21   1022.1426  1018.2819  0.0907  0.0812  1200.7059  1224.1646
c108_21   1010.7038  1003.3857  0.0816  0.0808  1234.2410  1179.8580
c109_21    980.2930   956.4030  0.0649  0.0565  1228.6510  1220.0588
c201_21    629.9494   629.9494  0.1553  0.1553  3359.4904  3359.4904

Notebook tiếp theo (`03_budget_curve_hypervolume.ipynb`) sẽ dùng `td_solutions.csv` và `solutions_clean.csv` để tính budgeted equity curve và Hypervolume 2D/3D.